In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch_numopt
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, TensorDataset
from sklearn.datasets import *
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error
from train_loop import train_loop

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [3]:
class Net(nn.Module):
    def __init__(self, input_size, device="cpu"):
        super().__init__()
        self.f1 = nn.Linear(input_size, 10, device=device)
        self.f2 = nn.Linear(10, 20, device=device)
        self.f3 = nn.Linear(20, 20, device=device)
        self.f4 = nn.Linear(20, 10, device=device)
        self.f5 = nn.Linear(10, 1, device=device)

        self.activation = nn.ReLU()
        # self.activation = nn.Sigmoid()

    def forward(self, x):
        x = self.activation(self.f1(x))
        x = self.activation(self.f2(x))
        x = self.activation(self.f3(x))
        x = self.activation(self.f4(x))
        x = self.f5(x)

        return x

In [4]:
# X, y = load_diabetes(return_X_y=True, scaled=False)
# X, y = make_regression(n_samples=1000, n_features=100)
X, y = make_friedman1(n_samples=1000, noise=1e-2)

X_scaler = MinMaxScaler()
X = X_scaler.fit_transform(X)
X = torch.Tensor(X).to(device)

y_scaler = MinMaxScaler()
y = y_scaler.fit_transform(y.reshape((-1, 1)))
y = torch.Tensor(y).to(device)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

torch_data = TensorDataset(X_train, y_train)
data_loader = DataLoader(torch_data, batch_size=10000)

torch.Size([800, 10]) torch.Size([800, 1])
torch.Size([200, 10]) torch.Size([200, 1])


In [5]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.LM(
    model=model,
    fletcher=False,
)

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=1000,
    max_patience=50
)

epoch:  0, loss: 0.0537298284471035
epoch:  1, loss: 0.04096207395195961
epoch:  2, loss: 0.03679712861776352
epoch:  3, loss: 0.036026787012815475
epoch:  4, loss: 0.03345722705125809
epoch:  5, loss: 0.032062970101833344
epoch:  6, loss: 0.028194718062877655
epoch:  7, loss: 0.025084104388952255
epoch:  8, loss: 0.021669507026672363
epoch:  9, loss: 0.01867014355957508
epoch:  10, loss: 0.017705125734210014
epoch:  11, loss: 0.016470475122332573
epoch:  12, loss: 0.015712609514594078
epoch:  13, loss: 0.01524564903229475
epoch:  14, loss: 0.014841501601040363
epoch:  15, loss: 0.01400385145097971
epoch:  16, loss: 0.01352076418697834
epoch:  17, loss: 0.013230320997536182
epoch:  18, loss: 0.012973872944712639
epoch:  19, loss: 0.012328430078923702
epoch:  20, loss: 0.012014557607471943
epoch:  21, loss: 0.011839382350444794
epoch:  22, loss: 0.011508924886584282
epoch:  23, loss: 0.011100671254098415
epoch:  24, loss: 0.01091169286519289
epoch:  25, loss: 0.010804479010403156
epoch:

In [6]:
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.9774587750434875
Test metrics:  R2 = 0.96157306432724


In [9]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.LM(
    model=model,
    fletcher=True,
    solver="pinv"
)

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=1000,
    max_patience=50
)

epoch:  0

/home/eugeniolr/Documents/Eugenio/torch_numopt/src/torch_numopt/utils.py:48: UserWarning: torch.linalg.svd: During SVD computation with the selected cusolver driver, batches 0 failed to converge. A more accurate method will be used to compute the SVD as a fallback. Check doc at https://pytorch.org/docs/stable/generated/torch.linalg.svd.html (Triggered internally at /pytorch/aten/src/ATen/native/cuda/linalg/BatchLinearAlgebraLib.cpp:690.)
  U, S, Vt = torch.linalg.svd(mat)


, loss: 0.10082259029150009
epoch:  1, loss: 0.045937489718198776
epoch:  2, loss: 0.03548760339617729
epoch:  3, loss: 0.03207472711801529
epoch:  4, loss: 0.02720581740140915
epoch:  5, loss: 0.01831972971558571
epoch:  6, loss: 0.015331153757870197
epoch:  7, loss: 0.013071889989078045
epoch:  8, loss: 0.01281053014099598
epoch:  9, loss: 0.012211775407195091
epoch:  10, loss: 0.011931740678846836
epoch:  11, loss: 0.011782166548073292
epoch:  12, loss: 0.011440299451351166
epoch:  13, loss: 0.011113413609564304
epoch:  14, loss: 0.010974650271236897
epoch:  15, loss: 0.010897098109126091
epoch:  16, loss: 0.010559567250311375
epoch:  17, loss: 0.01039762794971466
epoch:  18, loss: 0.010323775000870228
epoch:  19, loss: 0.010108590126037598
epoch:  20, loss: 0.00992717407643795
epoch:  21, loss: 0.00985985342413187
epoch:  22, loss: 0.009767163544893265
epoch:  23, loss: 0.009575104340910912
epoch:  24, loss: 0.009512592107057571
epoch:  25, loss: 0.009503944776952267
epoch:  26, lo

In [10]:
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.8718894720077515
Test metrics:  R2 = 0.8169565796852112
